In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss

In [2]:
def validation_step(df_y, winter, start_winter, chunksize=1):
    
    is_validation_data = (winter >= start_winter) & (winter < start_winter + chunksize)
    y_valid = df_y[is_validation_data].values[:,0]
    y_train = df_y[~is_validation_data].values[:,0]

    y_climpred = np.zeros((chunksize*84, 4))
    for i in range(4):
        y_climpred[:,i] = np.count_nonzero(y_train == i)/len(y_train)
    bs_clim = brier_score_loss(y_valid, y_climpred, labels=[0,1,2,3])
    
    return bs_clim

In [3]:
def cross_validation_k_fold(df_y, k=60, chunksize=1):

    year = df_y.index.year
    month = df_y.index.month
    winter = year.where(month < 12, year+1)
    start_winters = np.arange(winter[0], winter[-1]+1, chunksize)

    bs_k_fold = np.zeros(k)
    
    for i in range(k):
        bs_k_fold[i] = validation_step(df_y, winter, start_winters[i], chunksize)
        
    return bs_k_fold.mean()

In [4]:
df_target = pd.read_csv('targets/weather_regimes_target.csv')
df_target['time'] = pd.to_datetime(df_target['time'])
df_target = df_target.set_index('time')
df_target_train = df_target[df_target.index < "2010-07"]
cross_validation_k_fold(df_target_train)

np.float64(0.750627033905905)